In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import numpy as np

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

In [ ]:

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, 6),
            nn.ReLU(),
            nn.Linear(6, 3)
            n.Sigmoid() 
        )

    def forward(self, x):
        return self.net(x)
    
class Autoencoder(nn.Module):
    def __init__(self, encoding_dim=2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(4, 3),
            nn.ReLU(),
            nn.Linear(3, encoding_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(encoding_dim, 3),
            nn.ReLU(),
            nn.Linear(3, 4)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

In [ ]:
iris = datasets.load_iris()
X = iris.data
y = iris.target

In [17]:
print(X.shape)

(150, 4)


In [7]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [41]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.7, random_state=42)

In [42]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

In [10]:
mlp = MLP()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(mlp.parameters(), lr=0.01)

for epoch in range(100):
    optimizer.zero_grad()
    outputs = mlp(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()

In [11]:
with torch.no_grad():
    preds = mlp(X_test_t).argmax(dim=1)
mlp_acc = accuracy_score(y_test, preds.numpy())

In [27]:
autoencoder = Autoencoder(encoding_dim=2)
criterion = nn.MSELoss()
optimizer = optim.Adam(autoencoder.parameters(), lr=0.001, weight_decay=1e-5)

for epoch in range(2500):
    optimizer.zero_grad()
    outputs = autoencoder(X_train_t)
    loss = criterion(outputs, X_train_t)
    loss.backward()
    optimizer.step()

In [ ]:
with torch.no_grad():
    X_train_encoded = autoencoder.encoder(X_train_t).numpy()
    X_test_encoded = autoencoder.encoder(X_test_t).numpy()

In [ ]:
svm = SVC(kernel='rbf', gamma='scale')
svm.fit(X_train_encoded, y_train)
y_pred = svm.predict(X_test_encoded)
svm_acc = accuracy_score(y_test, y_pred)

In [ ]:
print(f"Dokładność MLP end-to-end: {mlp_acc:.3f}")
print(f"Dokładność SVM na cechach z autoenkodera: {svm_acc:.3f}")

Dokładność MLP end-to-end: 1.000
Dokładność SVM na cechach z autoenkodera: 0.300


In [ ]:
import numpy as np
import torch
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test, dtype=torch.float32)

autoencoder.eval()
with torch.no_grad():
    X_train_rec_t = autoencoder(X_train_t)
    X_test_rec_t  = autoencoder(X_test_t)

X_train_rec = X_train_rec_t.cpu().numpy()
X_test_rec  = X_test_rec_t.cpu().numpy()

mse_train = mean_squared_error(X_train, X_train_rec)
mse_test  = mean_squared_error(X_test, X_test_rec)
print(f"MSE train: {mse_train:.6f}, MSE test: {mse_test:.6f}")

per_feature_mse = ((X_test - X_test_rec)**2).mean(axis=0)
per_feature_r2  = [r2_score(X_test[:,i], X_test_rec[:,i]) for i in range(X_test.shape[1])]
print("Per-feature MSE:", np.round(per_feature_mse,6))
print("Per-feature R2:", np.round(per_feature_r2,4))

with torch.no_grad():
    X_train_latent = autoencoder.encoder(X_train_t).cpu().numpy()
    X_test_latent  = autoencoder.encoder(X_test_t).cpu().numpy()

print("Latent shape:", X_train_latent.shape)


log = LogisticRegression(max_iter=100)
log.fit(X_train_latent, y_train)
y_pred_log = log.predict(X_test_latent)
acc_log = accuracy_score(y_test, y_pred_log)

# svm
svm = SVC(kernel='rbf', gamma='scale')
svm.fit(X_train_latent, y_train)
y_pred_svm = svm.predict(X_test_latent)
acc_svm = accuracy_score(y_test, y_pred_svm)

print(f"Downstream accuracy (Logistic on latent): {acc_log:.3f}")
print(f"Downstream accuracy (SVM on latent): {acc_svm:.3f}")

from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train_raw = sc.fit_transform(X_train)
X_test_raw  = sc.transform(X_test)

log_raw = LogisticRegression(max_iter=100).fit(X_train_raw, y_train)
acc_log_raw = accuracy_score(y_test, log_raw.predict(X_test_raw))
print(f"Logistic on raw features: {acc_log_raw:.3f}")

if mse_test > 5*mse_train:
    print("Uwaga: możliwy overfitting (mse_test >> mse_train).")
if acc_svm < acc_log_raw:
    print("Enkoder daje gorsze cechy niż surowe dane dla SVM — rozważ zmiany w trainingu/architekturze.")


MSE train: 0.054816, MSE test: 0.037600
Per-feature MSE: [0.065985 0.010883 0.016595 0.056936]
Per-feature R2: [0.9354 0.9902 0.9842 0.9451]
Latent shape: (45, 2)
Downstream accuracy (Logistic on latent): 0.943
Downstream accuracy (SVM on latent): 0.914
Logistic on raw features: 0.943
Enkoder daje gorsze cechy niż surowe dane dla SVM — rozważ zmiany w trainingu/architekturze.
